# Setup

In [1]:
from shapely.geometry import box
import geopandas as gpd
import leafmap
import pandas as pd
import geoai
import torch
import glob
import rasterio


In [2]:
# the Syke peat after-use layer
peat = gpd.read_file("../data/raw/vectors/turvetuotantoalueet/turvetuotantoalueet_jalkikaytto.shp")

# leafmap needs WGS84 for display
peat_wgs = peat.to_crs("EPSG:4326")

# one colour per after-use class
colors = {
    "avoin": "#ff6600",
    "metsittynyt": "#00e64d",
    "turvetuotanto": "#cc00ff",
    "vesi": "#00ccff",
    "maatalous": "#ffee00",
    "rakennettu": "#ff0033",}

# same colours with English labels for the legend
legend = {
    "Open": "#ff6600",
    "Afforested": "#00e64d",
    "Still in production": "#cc00ff",
    "Water": "#00ccff",
    "Agriculture": "#ffee00",
    "Built-up": "#ff0033",}

In [3]:
# English labels for the Finnish class names
class_names_en = {
    "avoin": "open",
    "metsittynyt": "afforested",
    "turvetuotanto": "still_in_production",
    "vesi": "water",
    "maatalous": "agriculture",
    "rakennettu": "built_up",
}

peat_wgs["class_en"] = peat_wgs["luokka"].map(class_names_en)


# Inspection of training data

In [4]:
# slide a 0.4 x 0.3 degree window over the region and count what falls inside
results = []

for west in [21.8, 22.0, 22.2, 22.4, 22.6, 22.8, 23.0, 23.2, 23.4, 23.6, 23.8]:
    for south in [62.0, 62.2, 62.4, 62.6, 62.8, 63.0]:
        east = west + 0.4
        north = south + 0.3
        window = peat_wgs.cx[west:east, south:north]
        counts = window["class_en"].value_counts()
        results.append({
            "west": west,
            "south": south,
            "still_in_production": counts.get("still_in_production", 0),
            "open": counts.get("open", 0),
            "total": len(window),
        })

candidates = pd.DataFrame(results)
print(candidates.sort_values("still_in_production", ascending=False).head(15))

    west  south  still_in_production  open  total
19  22.4   62.2                  193  1576   2399
43  23.2   62.2                  192  2349   3779
37  23.0   62.2                  176  2558   4229
2   21.8   62.4                  171   696   1155
25  22.6   62.2                  154  1990   2914
24  22.6   62.0                  149  2177   3549
26  22.6   62.4                  139  1548   2331
20  22.4   62.4                  138  1062   1579
44  23.2   62.4                  122  1955   3294
18  22.4   62.0                  119  1075   1893
12  22.2   62.0                  117  1120   1757
50  23.4   62.4                  113  1400   2116
32  22.8   62.4                  112  1440   2249
31  22.8   62.2                  112  1853   2857
13  22.2   62.2                  112   715   1198


In [5]:
# training region 01, from the window search, 193 production sites
train_region01_2021 = {"west": 22.4, "south": 62.2, "east": 22.8, "north": 62.5}

# training region 02, 171 production sites
train_region02_2021 = {"west": 21.8, "south": 62.4, "east": 22.2, "north": 62.7}

# test region01, 192 production sites
test_region01_2021 = {"west": 23.2, "south": 62.2, "east": 23.6, "north": 62.5}

train_box_01 = gpd.GeoDataFrame(
    {"name": ["train01"]},
    geometry=[box(train_region01_2021["west"], train_region01_2021["south"], train_region01_2021["east"], train_region01_2021["north"])],
    crs="EPSG:4326",)

train_box_02 = gpd.GeoDataFrame(
    {"name": ["train02"]},
    geometry=[box(train_region02_2021["west"], train_region02_2021["south"], train_region02_2021["east"], train_region02_2021["north"])],
    crs="EPSG:4326",)

test_box_01 = gpd.GeoDataFrame(
    {"name": ["test01"]},
    geometry=[box(test_region01_2021["west"], test_region01_2021["south"], test_region01_2021["east"], test_region01_2021["north"])],
    crs="EPSG:4326",)

# regions added to the legend
legend_with_boxes = dict(legend)
legend_with_boxes["Train region 01"] = "#ffffff"
legend_with_boxes["Train region 02"] = "#0000ff"
legend_with_boxes["Test region"] = "#000000"

# only the polygons near the boxes, drawing all of Finland would hang the browser
peat_near = peat_wgs.cx[21.6:24.0, 62.0:63.3]

m = leafmap.Map(center=[62.5, 22.7], zoom=9)

# The layer "rakennettu" is left out as there is only one polygon in the whole dataset.
# one layer per class, each gets its own toggle in the layer control
m.add_gdf(peat_near[peat_near["luokka"] == "avoin"], layer_name="Open",
          style={"color": "#ff6600", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "metsittynyt"], layer_name="Afforested",
          style={"color": "#00e64d", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "turvetuotanto"], layer_name="Still in production",
          style={"color": "#cc00ff", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "vesi"], layer_name="Water",
          style={"color": "#00ccff", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "maatalous"], layer_name="Agriculture",
          style={"color": "#ffee00", "weight": 2, "fill": False})

# the three regions on top
m.add_gdf(train_box_01, layer_name="Train region 01", style={"color": "#ffffff", "weight": 3, "fill": False})
m.add_gdf(train_box_02, layer_name="Train region 02", style={"color": "#0000ff", "weight": 3, "fill": False})
m.add_gdf(test_box_01, layer_name="Test region 01", style={"color": "#000000", "weight": 3, "fill": False})

m.add_legend(title="After-use class", legend_dict=legend_with_boxes)
m

Map(center=[62.5, 22.7], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [6]:
train_region01 = peat_wgs.cx[22.4:22.8, 62.2:62.5]
train_region02 = peat_wgs.cx[21.8:22.2, 62.4:62.7]
test_region01 = peat_wgs.cx[23.2:23.6, 62.2:62.5]

print("TRAIN_Region01")
print(train_region01["class_en"].value_counts())

print("TRAIN_Region02")
print(train_region02["class_en"].value_counts())

print("TEST_Region01")
print(test_region01["class_en"].value_counts())

TRAIN_Region01
class_en
open                   1576
afforested              388
still_in_production     193
agriculture             125
water                   117
Name: count, dtype: int64
TRAIN_Region02
class_en
open                   696
still_in_production    171
afforested             127
agriculture             96
water                   65
Name: count, dtype: int64
TEST_Region01
class_en
open                   2349
afforested             1084
still_in_production     192
agriculture              80
water                    74
Name: count, dtype: int64


In [7]:
# Checking the raster data for the training region01
geoai.print_raster_info("../data/raw/raster/train_region01_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/train_region01_2021.tif =====
Driver: GTiff
Dimensions: 2155 x 3394 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=572110.0, bottom=6897240.0, right=593660.0, top=6931180.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 45.00
  Max: 12708.00
  Mean: 310.46
  Std Dev: 133.98
Band 2:
  Min: 64.00
  Max: 12756.00
  Mean: 480.11
  Std Dev: 181.66
Band 3:
  Min: 49.00
  Max: 10376.00
  Mean: 369.77
  Std Dev: 224.01
Band 4:
  Min: 38.00
  Max: 12124.00
  Mean: 2537.19
  Std Dev: 831.95
Band 5:
  Min: 25.00
  Max: 6211.00
  Mean: 1567.23
  Std Dev: 612.32
Band 6:
  Min: 14.00
  Max: 5990.00
  Mean: 859.62
  Std Dev: 486.57


In [8]:
# Checking the raster data for the training region02
geoai.print_raster_info("../data/raw/raster/train_region02_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/train_region02_2021.tif =====
Driver: GTiff
Dimensions: 2110 x 3375 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=540930.0, bottom=6918990.0, right=562030.0, top=6952740.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 108.00
  Max: 9160.00
  Mean: 336.67
  Std Dev: 157.96
Band 2:
  Min: 89.00
  Max: 8756.00
  Mean: 515.17
  Std Dev: 201.34
Band 3:
  Min: 62.00
  Max: 9064.00
  Mean: 389.56
  Std Dev: 246.56
Band 4:
  Min: 70.00
  Max: 9824.00
  Mean: 2730.69
  Std Dev: 750.01
Band 5:
  Min: 45.00
  Max: 6191.00
  Mean: 1581.82
  Std Dev: 586.15
Band 6:
  Min: 27.00
  Max: 6111.00
  Mean: 863.34
  Std Dev: 469.23


In [9]:
# Checking if production sites look different from open sites
peat_aoi = peat_wgs.cx[22.4:22.8, 62.2:62.5]

m = leafmap.Map(center=[62.35, 22.6], zoom=11)
m.add_raster("../data/raw/raster/train_ndvi_2021.tif", colormap="RdYlGn", vmin=0, vmax=0.9, layer_name="NDVI")

for cls, c in colors.items():
    sub = peat_aoi[peat_aoi["luokka"] == cls]
    if len(sub) > 0:
        m.add_gdf(sub, layer_name=cls, style={"color": c, "weight": 2, "fill": False})

m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.349972, 22.6020605], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [11]:
# Checking polygons by area rather than polygon count
region01 = peat_wgs.cx[22.4:22.8, 62.2:62.5].to_crs(3067)
region01["area_ha"] = region01.area / 10000

region02 = peat_wgs.cx[21.8:22.2, 62.4:62.7].to_crs(3067)
region02["area_ha"] = region02.area / 10000

test01 = peat_wgs.cx[23.2:23.6, 62.2:62.5].to_crs(3067)
test01["area_ha"] = test01.area / 10000

print("TRAIN_Region01 area in hectares:")
print(region01.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

print("TRAIN_Region02 area in hectares:")
print(region02.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

print("TEST_Region01 area in hectares:")
print(test01.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

TRAIN_Region01 area in hectares:
class_en
still_in_production    2288.969268
agriculture            1739.232797
open                    582.307073
afforested               79.734218
water                    16.199157
Name: area_ha, dtype: float64
TRAIN_Region02 area in hectares:
class_en
still_in_production    1847.385590
agriculture             534.275649
open                    155.460795
afforested               25.996707
water                    21.915044
Name: area_ha, dtype: float64
TEST_Region01 area in hectares:
class_en
still_in_production    1303.078343
open                    657.139798
agriculture             425.354479
afforested              117.506111
water                    47.522672
Name: area_ha, dtype: float64


In [4]:
# production polygons clipped to the training region01, in the raster's CRS
train_labels = peat_wgs.cx[22.4:22.8, 62.2:62.5]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/train_labels_binary/train_labels_binary_region01_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

193 polygons, CRS EPSG:32634


In [5]:
# production polygons clipped to the training region02, in the raster's CRS
train_labels = peat_wgs.cx[21.8:22.2, 62.4:62.7]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/train_labels_binary/train_labels_binary_region02_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

171 polygons, CRS EPSG:32634


In [14]:
# production polygons clipped to the test region01, in the raster's CRS
test_labels = peat_wgs.cx[23.2:23.6, 62.2:62.5]
test_labels = test_labels[test_labels["class_en"] == "still_in_production"]
test_labels = test_labels.to_crs("EPSG:32634")

test_labels.to_file("../data/processed/test_labels_binary_region01_2021.gpkg", driver="GPKG")
print(len(test_labels), "polygons, CRS", test_labels.crs)

192 polygons, CRS EPSG:32634


# Tiling the Imagery

In [ ]:
binary_train_raster_path_2021 = "../data/raw/raster/binary_model_train_2021.tif"
binary_train_vector_path_2021 = "../data/processed/binary_model_train_labels_2021.gpkg"

tiles = geoai.export_geotiff_tiles(
    in_raster=binary_train_raster_path_2021,
    out_folder=out_folder,
    in_class_data=binary_train_vector_path_2021,
    tile_size=256,
    stride=128,
    buffer_radius=0,
)

'class' not found in vector data. Using default class ID 1.


Generating tiles:   0%|          | 0/416 [00:00<?, ?it/s]

In [13]:
out_folder = "../data/training/binary_model_train_tiles"

In [14]:
labels = sorted(glob.glob(f"{out_folder}/labels/*.tif"))
images = sorted(glob.glob(f"{out_folder}/images/*.tif"))
print(len(images), "images,", len(labels), "labels")

maxes = []
for path in labels:
    with rasterio.open(path) as src:
        maxes.append(int(src.read(1).max()))

print("highest value in any label:", max(maxes))
print("labels above 1:", sum(m > 1 for m in maxes))

416 images, 416 labels
highest value in any label: 255
labels above 1: 363


In [15]:
counts = []
for path in labels:
    with rasterio.open(path) as src:
        counts.append(int(src.read(1).sum()))

print("tiles with something:", sum(c > 0 for c in counts))
print("total positive pixels:", sum(counts))

tiles with something: 241
total positive pixels: 959742


# Runing the binary_model

In [11]:
from pathlib import Path

In [ ]:
output_dir = "../models/binary_model_unet_2021"
print(Path(output_dir).resolve())

C:\Users\clair\OneDrive\Dokumente\University Zurich\Semester 7 (HS)\sds320\project\models\binary_model_unet_2021


In [17]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=output_dir,
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=6,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
)

# Evaluating the binary Model

In [13]:
history = torch.load("../models/binary_model_unet_2021/training_history.pth", weights_only=False)
print(type(history))
print(history.keys() if hasattr(history, "keys") else history[:2])

<class 'dict'>
dict_keys(['train_losses', 'val_losses', 'val_ious', 'val_f1s', 'val_precisions', 'val_recalls'])


In [14]:
history = torch.load("../models/binary_model_unet_2021/training_history.pth", weights_only=False)

df = pd.DataFrame({
    "epoch": range(1, len(history["train_losses"]) + 1),
    "train_loss": history["train_losses"],
    "val_loss": history["val_losses"],
    "val_iou": history["val_ious"],
    "val_f1": history["val_f1s"],
    "val_precision": history["val_precisions"],
    "val_recall": history["val_recalls"],
})

df

,epoch,train_loss,val_loss,val_iou,val_f1,val_precision,val_recall
0,1,0.706641,0.836374,0.355089,0.486690,0.555776,0.575144
1,2,0.681956,0.810924,0.354065,0.483705,0.654082,0.595179
2,3,0.682355,0.809459,0.370740,0.501847,0.636934,0.608833
3,4,0.674056,0.800232,0.401799,0.542140,0.632368,0.623408
4,5,0.676515,0.648598,0.422683,0.550458,0.643277,0.648772
5,6,0.678223,0.876916,0.274574,0.374569,0.594174,0.529267
6,7,0.676436,0.696894,0.371345,0.463009,0.649492,0.601396
7,8,0.674570,0.644568,0.459540,0.608021,0.644885,0.690277
8,9,0.667296,0.618991,0.484380,0.620903,0.672932,0.700898
9,10,0.668132,0.733054,0.378210,0.528329,0.572132,0.604729


In [15]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=5, subplot_titles=["Loss", "IoU", "F1", "Precision", "Recall"])

fig.add_trace(go.Scatter(x=df["epoch"], y=df["train_loss"], name="Train Loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_loss"], name="Val Loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_iou"], name="Val IoU"), row=1, col=2)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_f1"], name="Val F1"), row=1, col=3)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_precision"], name="Val Precision"), row=1, col=4)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_recall"], name="Val Recall"), row=1, col=5)

fig.update_layout(height=350, width=1400)
fig.show()

In [17]:
import pandas as pd

In [18]:
history = torch.load(
    "../models/binary_model_unet_2021/training_history.pth",
    weights_only=False,
)

history_df = pd.DataFrame(history)
history_df.insert(0, "epoch", range(1, len(history_df) + 1))
print(history_df)

    epoch  train_losses  val_losses  val_ious   val_f1s  val_precisions  \
0       1      0.706641    0.836374  0.355089  0.486690        0.555776   
1       2      0.681956    0.810924  0.354065  0.483705        0.654082   
2       3      0.682355    0.809459  0.370740  0.501847        0.636934   
3       4      0.674056    0.800232  0.401799  0.542140        0.632368   
4       5      0.676515    0.648598  0.422683  0.550458        0.643277   
5       6      0.678223    0.876916  0.274574  0.374569        0.594174   
6       7      0.676436    0.696894  0.371345  0.463009        0.649492   
7       8      0.674570    0.644568  0.459540  0.608021        0.644885   
8       9      0.667296    0.618991  0.484380  0.620903        0.672932   
9      10      0.668132    0.733054  0.378210  0.528329        0.572132   
10     11      0.664440    0.653998  0.418895  0.543761        0.628803   
11     12      0.667402    0.633632  0.455433  0.589677        0.633398   
12     13      0.669539  

In [22]:
import plotly.express as px

fig = px.line(history_df, x="epoch", y="train_losses", title="Train loss")
fig.show()

fig = px.line(history_df, x="epoch", y="val_losses", title="Validation loss")
fig.show()

fig = px.line(history_df, x="epoch", y="val_ious", title="Validation IoU")
fig.show()

fig = px.line(history_df, x="epoch", y="val_f1s", title="Validation F1")
fig.show()

fig = px.line(history_df, x="epoch", y="val_precisions", title="Validation precision")
fig.show()

fig = px.line(history_df, x="epoch", y="val_recalls", title="Validation recall")
fig.show()

# Binary segmentation Model run 2 (removal of NoData and lowering the learning rate)